In [ ]:
import copy
import os.path as osp

import numpy as np
import pandas as pd
from calisim.data_model import (
	DistributionModel,
	ParameterDataType,
	ParameterSpecification,
)
from calisim.history_matching import (
	HistoryMatchingMethod,
	HistoryMatchingMethodModel,
)
from pcse.base import ParameterProvider
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from pcse.models import Wofost72_PP

In [ ]:
wdp = NASAPowerWeatherDataProvider(latitude=52, longitude=5)
print(wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
agro = YAMLAgroManagementReader(osp.join("..", "data", "AGMT_C2_2020.agro"))
print(agro)

In [ ]:
params = ParameterProvider(cropdata=cropd, sitedata=sited, soildata=soild)
wofost = Wofost72_PP(params, wdp, agro)
wofost.run_till_terminate()
observed_data = pd.DataFrame(wofost.get_output())
observed_data

In [ ]:
parameter_spec = ParameterSpecification(
	parameters=[
		DistributionModel(
			name="TDWI",
			distribution_name="uniform",
			distribution_args=[75, 700],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="SPAN",
			distribution_name="uniform",
			distribution_args=[20, 50],
			data_type=ParameterDataType.CONTINUOUS,
		),
	]
)

In [ ]:
def history_matching_func(
	parameters: dict, simulation_id: str, observed_data: np.ndarray | None
) -> float | list[float]:
	p = copy.deepcopy(params)
	for k in parameters:
		p.set_override(k, parameters[k])

	wofost = Wofost72_PP(p, wdp, agro)
	wofost.run_till_terminate()
	simulated_data = pd.DataFrame(wofost.get_output()).LAI.values
	return simulated_data

In [ ]:
specification = HistoryMatchingMethodModel(
	experiment_name="ies_history_matching",
	parameter_spec=parameter_spec,
	observed_data=observed_data.LAI.values,
	method="sies",
	n_samples=25,
	n_iterations=10,
	output_labels=["Leaf Area Index"],
	verbose=True,
	batched=False,
	covariance=np.eye(observed_data.LAI.values.shape[0]),
	method_kwargs=dict(truncation=1.0),
)

calibrator = HistoryMatchingMethod(
	calibration_func=history_matching_func, specification=specification, engine="ies"
)

calibrator.specify().execute().analyze()